[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/shells/shells.ipynb)

# Discrete Shells

This example computes discrete shells, after Grinspun, Hirani, Desbrun and Schröder, "Discrete shells" (2003): thin elastic sheets modelled as triangle meshes. A thin sheet resists two things, stretching, which changes lengths along it, and bending, which changes how it curves; the mesh stretches along its edges and bends at its hinges, the edges with a face on either side. A sheet whose rest state does not fit in space must buckle: a disc that grows towards its rim becomes a saddle, and a strip with the crossed natural curvature of a seed pod's two layers twists into a ribbon.

In the projective geometric algebra of space an edge is the line joining its two ends, `tails & heads`, whose norm is its length, and a hinge's two faces lie in the planes joining its line with the corners across it. The motor that folds one face onto the other is the square root of their product, `(second.normalized() * first.normalized()).square_root()`, and its logarithm is half the bending angle times the hinge's unit line. The sheet's quantities are fields over the mesh's vertices, edges, faces and hinges, and the reverses of the mesh's reading maps, such as `~sheet.mesh.d[0] * pull`, gather the edges' and hinges' forces at the vertices. As internal forces they balance: the lines they act along add up to no force and no moment.

In the notation of the discrete shells paper the bending energy is $\sum_e (\theta_e - \bar\theta_e)^2 \, 3 \bar\ell_e^2 / \bar A_e$, with $\bar A_e$ the summed area of the edge's two faces.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from dataclasses import dataclass
from itertools import tee

import jax
import numpy as np

from numga import stack
from numga.backend.jax import JaxContext
from numga.algebras import PGA3D as ga
from numga.sparse import SparseExtensor
from examples.util.animation import save_figure, show_animation
from examples.mechanics.shells import render
from examples.mechanics.shells.core import flat_mesh, lattice
from examples import instantiate

np.set_printoptions(precision=3, suppress=True)

Scalar = ga.gatype.scalar()                                  # Scalar+
Plane = ga.gatype.vector()                                   # Vector+
Point = ga.gatype.antivector()                               # Antivector-
Direction = Point.derive.degenerate                          # Antivector-
# Lines joining points: edges, hinges and the lines forces act along.
Line = ga.gatype.antibivector()                              # Antibivector-
Motor = ga.gatype.rotor()                                    # Rotor+
# The form of a natural curvature, on unit lines in the sheet: the curvature across them.
Curvature = ga.gatype((Scalar, Line, Line))                  # Scalar+ <- (Line-, Line-)

mesh_module = instantiate("examples.util.mesh", ga)
Mesh, as_scalar = mesh_module.Mesh, mesh_module.as_scalar
mv = mesh_module.ctx.multivector

## 1. Energy and forces

The sheet stores energy as the squared stretch of every edge, against its rest length, and the squared bend of every hinge, against its rest angle. The forces are the energy's exact downhill direction. Every edge pulls its ends together or apart along its line. A hinge bent away from its rest angle twists its two faces about its line, equally and oppositely: each face pushes the corner across the hinge square to the face, by the twist over the corner's distance from the hinge, `(plane | corner) ^ mv.w` being the direction of the line through the corner square to the plane, and the hinge's two ends take the opposite push, shared by the lever rule, by where the corner lies along the hinge. As internal forces they balance: the lines along which they act, the joins of each vertex with its force, add up to a forque, a force and a moment at once, and for the sheet's own forces it is zero.

In the notation of the discrete shells paper the gradient of the bending angle at the corner across the hinge is $n / h$, the unit normal over the corner's height above the hinge.

In [ ]:
@dataclass(frozen=True)
class Sheet:
    """A shell's rest state on a mesh: the maps reading every hinge's two ends and the corners across it
    `[4] Scalar+[hinges <- vertices]`, every edge's rest length `Scalar+[edges]` and every hinge's rest angle
    `Scalar+[hinges]`, the stiffness of both, and the mass at every vertex `Scalar+[vertices]`."""

    mesh: Mesh
    corners: SparseExtensor
    lengths: Scalar
    angles: Scalar
    stretching: float
    bending: Scalar
    masses: Scalar


def energy(sheet: Sheet, points: Point) -> Scalar:
    tails, heads = sheet.mesh.incidence[1, 0] * points
    stretched = (tails & heads).norm() - sheet.lengths                           # Scalar+[edges]
    ends, tips, across, opposite = sheet.corners * points                        # Point-[hinges], Point-[hinges], Point-[hinges], Point-[hinges]
    line = ends & tips                                                           # Line-[hinges]
    # Both planes wound as the first face, so that a flat hinge folds by nothing.
    first, second = line & across, -(line & opposite)                            # Plane+[hinges], Plane+[hinges]
    # The product of two unit planes is twice the turn about their common line: its square root folds one onto
    # the other, and twice its logarithm, along the unit line, is the bending angle.
    motor = (second.normalized() * first.normalized()).square_root()             # Motor+[hinges]
    bent = -2 * (motor.log() | line.normalized()) - sheet.angles                 # Scalar+[hinges]
    stretching = (sheet.stretching * stretched * stretched / sheet.lengths).sites.sum()   # [] Scalar+
    bending = (sheet.bending * bent * bent).sites.sum()                          # [] Scalar+
    return (stretching + bending) / 2


def forces(sheet: Sheet, points: Point) -> Direction:
    # The edges' springs pulling their ends together or apart.
    tails, heads = sheet.mesh.incidence[1, 0] * points
    lengths = (tails & heads).norm()                                             # Scalar+[edges]
    pull = (heads - tails) * (sheet.stretching * (lengths - sheet.lengths) / (sheet.lengths * lengths))   # Direction-[edges]

    # The hinges twisting their faces back towards their rest angles.
    ends, tips, across, opposite = sheet.corners * points                        # Point-[hinges], Point-[hinges], Point-[hinges], Point-[hinges]
    line = ends & tips                                                           # Line-[hinges]
    first, second = line & across, -(line & opposite)                            # Plane+[hinges], Plane+[hinges]
    motor = (second.normalized() * first.normalized()).square_root()             # Motor+[hinges]
    moment = line.norm() * sheet.bending * (-2 * (motor.log() | line.normalized()) - sheet.angles)   # Scalar+[hinges]
    # Each face pushes the corner across the hinge square to its plane, along `(plane | corner) ^ w`, by the
    # moment over the corner's distance from the hinge.
    pushes = [((plane | corner) ^ mv.w) * (moment / plane.norm_squared()) for plane, corner in ((first, across), (second, opposite))]
    # The hinge's ends take the opposite pushes, shared by the lever rule: by where each corner lies along the hinge.
    shares = [((ends & corner) | line) / (line | line) for corner in (across, opposite)]
    at_tips = -(pushes[0] * shares[0] + pushes[1] * shares[1])                   # Direction-[hinges]
    at_ends = -(pushes[0] + pushes[1]) - at_tips                                 # Direction-[hinges]

    # The incidence's reverse sums every edge's pull at its two ends, and each reading map's reverse sums
    # the hinges' pushes at their corners.
    return -(~sheet.mesh.d[0] * pull) + (~sheet.corners * stack([at_ends, at_tips, *pushes])).sum(axis=0)

## 2. The rest state, and relaxing towards it

The mesh knows every hinge, an edge between two faces, with the corners across it, `mesh.edge_opposites`. The rest angles come from a natural curvature, a form on lines: bound with a hinge's unit line on both sides it gives the curvature across the hinge, and times the distance between the two faces' centres, a third of the sum of the corners' distances from the hinge, the angle the hinge would bend by on a surface of that curvature. The bending angles count towards the sheet's top side, read off the planes through its faces. The sheet then moves under its own forces from nearly flat, losing a share of its velocity every step, until it settles.

In [ ]:
# The share of its velocity the sheet loses every step, the frames, and the steps per frame; the size and the
# seed of the random heights that let a flat sheet leave its plane.
damping = 0.01
n_frames, n_steps_per_frame = 100, 20
roughness, seed = 1e-3, 0

In [ ]:
def sheet(mesh: Mesh, points: Point, lengths: Scalar, curvature: Curvature, top: Plane, thickness: float) -> Sheet:
    # A sheet of the given thickness, of unit stiffness and density, at rest at the points of the mesh's vertices.
    # Every hinge's two ends, as the face on its left winds them, then the corner across it in that face and in
    # the other.
    hinge_vertices = np.concatenate([mesh.edges[mesh.interior_edges], mesh.edge_opposites], axis=-1)   # [hinges, 4]
    corners = mesh.reading(hinge_vertices.T)                                     # [4] Scalar+[hinges <- vertices]
    ends, tips, across, opposite = corners * points                              # Point-[hinges], Point-[hinges], Point-[hinges], Point-[hinges]
    line = ends & tips                                                           # Line-[hinges]
    first, second = line & across, -(line & opposite)                            # Plane+[hinges], Plane+[hinges]
    # The curvature across each hinge, times the distance between its faces' centres, a third of the sum of
    # the corners' distances from the hinge, is the angle the hinge bends by on a surface of that curvature.
    unit = line.normalized()
    angles = curvature(unit, unit) * (first.norm() + second.norm()) / (3 * line.norm())   # Scalar+[hinges]
    # The plane through each face's corners, wound as the face, has twice the face's area as its norm; read
    # against the top side, it says which way the bending angles count.
    face_corners = mesh.incidence[2, 0] * points                                 # [corners] Point-[faces]
    planes = face_corners[0] & face_corners[1] & face_corners[2]                 # Plane+[faces]
    facing = (planes | top).sites.sum()                                          # [] Scalar+
    # Each vertex carries a third of the mass of every face at it.
    double_areas = planes.norm()                                                 # Scalar+[faces]
    masses = ~mesh.incidence[2, 0].sum(axis=0) * double_areas * (thickness / 6)
    return Sheet(mesh=mesh, corners=corners, lengths=lengths, angles=angles * (facing / facing.norm()),
                 stretching=thickness, bending=thickness ** 3 / 2 * -(line | line) / (first.norm() + second.norm()),
                 masses=masses)


def relax(sheet: Sheet, points: Point, step: float, damping: float, n_frames: int, n_steps_per_frame: int):
    # The points at every frame, so many steps apart, and at the end, moving from rest under the sheet's own forces.
    def substep(state: tuple[Point, Direction], _) -> tuple[tuple[Point, Direction], None]:
        points, velocity = state
        velocity = (velocity + forces(sheet, points) * (step / sheet.masses)) * (1 - damping)
        return (points + velocity * step, velocity), None

    # The steps between two frames, compiled once into one loop.
    frame = jax.jit(lambda state: jax.lax.scan(substep, state, length=n_steps_per_frame)[0])
    context = JaxContext(ga)
    state = (points.materialize(context), (points - points).materialize(context))   # Point-[vertices], Direction-[vertices]
    for _ in range(n_frames):
        yield state[0].materialize(mv.context)
        state = frame(state)
    yield state[0].materialize(mv.context)

## 3. A disc that grows towards its rim

Every edge's rest length grows with the square of its middle's distance from the centre, by 30 percent at the rim. A flat disc has too little rim for those lengths, so the disc leaves its plane, and with no natural curvature it settles into a saddle. This cell takes about fifteen seconds, most of it the relaxation's two thousand steps.

In [ ]:
# The disc: its radius, the spacing of its lattice, its growth at the rim, its thickness, the time step, and the
# duration of a frame.
disc_radius, disc_spacing, growth, disc_thickness, disc_step = 1.0, 0.1, 0.3, 0.03, 0.02
duration_ms = 40

In [ ]:
flat = lattice(disc_spacing, disc_radius)
flat = flat[np.linalg.norm(flat, axis=-1) <= disc_radius + 1e-9]
disc_mesh, disc_points = flat_mesh(flat, disc_spacing)
tails, heads = disc_mesh.incidence[1, 0] * disc_points                         # Point-[edges], Point-[edges]
# Each edge grows with the square of its middle's distance from the centre, in disc radii.
distance = (mv.w.dual() & ((tails + heads) / 2)).norm() / disc_radius          # Scalar+[edges]
disc_lengths = (tails & heads).norm() * (1 + growth * distance * distance)     # Scalar+[edges]
disc = sheet(disc_mesh, disc_points, disc_lengths, 0 * (Line | Line), mv.z, disc_thickness)
# The flat disc lifted by small random heights, to let it leave its plane.
heights = roughness * np.random.default_rng(seed).normal(size=(len(flat), 1))
rough = mv.point(np.concatenate([flat, heights], axis=-1)).field()             # Point-[vertices]
shown, disc_frames = tee(relax(disc, rough, disc_step, damping, n_frames, n_steps_per_frame))
show_animation(render.animate(disc_mesh.faces, shown), "shells_disc", duration_ms)
disc_frames = list(disc_frames)

## 4. A seed pod's strip

A seed pod's wall is two layers that each want to curve along their own direction, the two crossing the strip at 45 degrees either way. The natural curvature curves positively across one direction and negatively across the other, a saddle that no unstretched surface can take. Neither layer wins along the whole strip, and it twists into a ribbon. This cell takes about a dozen seconds, most of it the relaxation's two thousand steps.

In [ ]:
# The strip: its length, width, the spacing of its lattice, its thickness and the time step; the layers' natural
# curvature, and the angle of their directions from the strip's axis.
strip_length, strip_width, strip_spacing, strip_thickness, strip_step = 2.0, 0.3, 0.08, 0.05, 0.012
natural_curvature, pod_angle = 6.0, np.pi / 4

In [ ]:
flat = lattice(strip_spacing, strip_length)
flat = flat[(np.abs(flat[:, 0]) <= strip_length / 2) & (np.abs(flat[:, 1]) <= strip_width / 2)]
strip_mesh, strip_points = flat_mesh(flat, strip_spacing)
tails, heads = strip_mesh.incidence[1, 0] * strip_points                # Point-[edges], Point-[edges]
# The two layers curve along lines through the centre, turned either way from the strip's axis.
axis = mv.wx.dual()                                                     # [] Line-
along, across = (((mv.xy * (-sense * pod_angle / 2)).exp() >> axis) for sense in (1, -1))
curvature = natural_curvature * ((across | Line) * (across | Line) - (along | Line) * (along | Line))   # [] Scalar+ <- (Line-, Line-)
strip = sheet(strip_mesh, strip_points, (tails & heads).norm(), curvature, mv.z, strip_thickness)
# The flat strip lifted by small random heights, to let it leave its plane.
heights = roughness * np.random.default_rng(seed).normal(size=(len(flat), 1))
rough = mv.point(np.concatenate([flat, heights], axis=-1)).field()      # Point-[vertices]
strip_frames = list(relax(strip, rough, strip_step, damping, n_frames, n_steps_per_frame))
show_animation(render.animate(strip_mesh.faces, strip_frames), "shells_strip", duration_ms)

The settled disc and strip side by side, shaded by the directions their faces face.

In [ ]:
save_figure(render.draw_settled([(disc_mesh.faces, disc_frames[-1]), (strip_mesh.faces, strip_frames[-1])]), "shells_settled");

In [ ]:
# checks
for shell, frames, rise in ((disc, disc_frames, 0.3), (strip, strip_frames, 0.1)):
    settled = frames[-1]
    # The forces inside the sheet balance: their lines add up to no force and no moment.
    total = (settled & forces(shell, settled)).sites.sum()
    assert (-(total | total) - (total.dual() | total.dual())) < 1e-30
    # The sheet has left its plane.
    assert np.ptp((settled & mv.z).batch().to_array()) > rise
# The disc has settled close to its grown lengths.
tails, heads = disc_mesh.incidence[1, 0] * disc_frames[-1]
strain = ((tails & heads).norm() / disc_lengths - 1).batch().to_array()
assert np.median(np.abs(strain)) < 0.02